# Agente Strands com Observabilidade Arize no Amazon Bedrock AgentCore Runtime

## Visão Geral

Este notebook demonstra a implantação de um agente Strands no Amazon Bedrock AgentCore Runtime com integração de observabilidade Arize. A implementação utiliza modelos Amazon Bedrock Claude e envia dados de telemetria para o Arize através do OpenTelemetry (OTEL).

## Componentes Principais

- **Strands Agents**: Framework Python para construção de agentes alimentados por LLM com suporte integrado a telemetria
- **Amazon Bedrock AgentCore Runtime**: Serviço de runtime gerenciado para hospedagem e escalabilidade de agentes na AWS
- **Arize**: Plataforma de observabilidade para aplicações LLM que recebe traces via OTEL
- **OpenTelemetry**: Protocolo padrão da indústria para coleta e exportação de dados de telemetria

## Arquitetura

O agente é containerizado e implantado no AgentCore Runtime, que fornece endpoints HTTP para invocação. Os dados de telemetria fluem do agente Strands através dos exportadores OTEL para o Arize para monitoramento e depuração. A implementação desabilita a observabilidade padrão do AgentCore para usar o Arize em seu lugar.

## Pré-requisitos

- Python 3.10+
- Credenciais AWS configuradas com permissões para Bedrock e AgentCore
- Conta [Arize](https://app.arize.com/) com chaves de API e space
- Docker instalado localmente
- Acesso aos modelos Amazon Bedrock Claude em us-west-2

In [ ]:
!pip install --force-reinstall -U -r requirements.txt --quiet

## Configurar Credenciais AWS

## Implementação do Agente

O arquivo do agente (`strands_claude.py`) implementa um agente de viagens com capacidades de busca na web. A configuração principal inclui:
- Inicialização da telemetria do Strands

In [ ]:
# Arize configuration
import os
os.environ["ARIZE_API_KEY"] = ""    # <--- UPDATE WITH YOUR ARIZE API KEY 
os.environ["ARIZE_SPACE_ID"] = ""   # <--- UPDATE WITH YOUR ARIZE SPACE ID
os.environ["ARIZE_ENDPOINT"] = "https://otlp.arize.com:443"



In [ ]:
%%writefile strands_claude.py
import os
import logging
from bedrock_agentcore.runtime import BedrockAgentCoreApp
from strands import Agent, tool
from strands.models import BedrockModel
from strands.telemetry import StrandsTelemetry
from ddgs import DDGS
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import BatchSpanProcessor
from opentelemetry.sdk.resources import Resource
from opentelemetry.exporter.otlp.proto.grpc.trace_exporter import OTLPSpanExporter
from openinference.instrumentation.strands_agents import StrandsAgentsToOpenInferenceProcessor

#Arize Project Name
os.environ["ARIZE_PROJECT_NAME"] = "agentcore-strands-travel-agent" ## <-- Update with your Arize Project Name


strands_processor = StrandsAgentsToOpenInferenceProcessor()
resource = Resource.create({
  "model_id": os.environ["ARIZE_PROJECT_NAME"], 
})
provider = TracerProvider(resource=resource)
provider.add_span_processor(strands_processor)
otel_exporter = OTLPSpanExporter()
provider.add_span_processor(BatchSpanProcessor(otel_exporter))
trace.set_tracer_provider(provider)

logging.basicConfig(level=logging.ERROR, format="[%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)
logger.setLevel(os.getenv("AGENT_RUNTIME_LOG_LEVEL", "INFO").upper())


@tool
def web_search(query: str) -> str:
    """
    Search the web for information using DuckDuckGo.

    Args:
        query: The search query

    Returns:
        A string containing the search results
    """
    try:
        ddgs = DDGS()
        results = ddgs.text(query, max_results=5)

        formatted_results = []
        for i, result in enumerate(results, 1):
            formatted_results.append(
                f"{i}. {result.get('title', 'No title')}\n"
                f"   {result.get('body', 'No summary')}\n"
                f"   Source: {result.get('href', 'No URL')}\n"
            )

        return "\n".join(formatted_results) if formatted_results else "No results found."

    except Exception as e:
        return f"Error searching the web: {str(e)}"

# Function to initialize Bedrock model
def get_bedrock_model():
    region = os.getenv("AWS_DEFAULT_REGION", "us-west-2")
    model_id = os.getenv("BEDROCK_MODEL_ID", "us.anthropic.claude-3-7-sonnet-20250219-v1:0")

    bedrock_model = BedrockModel(
        model_id=model_id,
        region_name=region,
        temperature=0.0,
        max_tokens=1024
    )
    return bedrock_model

# Initialize the Bedrock model
bedrock_model = get_bedrock_model()

# Define the agent's system prompt
system_prompt = """You are an experienced travel agent specializing in personalized travel recommendations 
with access to real-time web information. Your role is to find dream destinations matching user preferences 
using web search for current information. You should provide comprehensive recommendations with current 
information, brief descriptions, and practical travel details."""

app = BedrockAgentCoreApp()

def initialize_agent():
    """Initialize the agent with proper telemetry configuration."""

    # Create and cache the agent
    agent = Agent(
        model=bedrock_model,
        system_prompt=system_prompt,
        tools=[web_search]
    )
    
    return agent

@app.entrypoint
def strands_agent_bedrock(payload, context=None):
    """
    Invoke the agent with a payload
    """
    user_input = payload.get("prompt")
    logger.info("[%s] User input: %s", context.session_id, user_input)
    
    # Initialize agent with proper configuration
    agent = initialize_agent()
    
    response = agent(user_input)
    return response.message['content'][0]['text']

if __name__ == "__main__":
    app.run()

### Configurar implantação do AgentCore Runtime

Em seguida, usaremos nosso starter toolkit para configurar a implantação do AgentCore Runtime com um entrypoint, a execution role que acabamos de criar e um arquivo de requirements. Também configuraremos o starter kit para criar automaticamente o repositório Amazon ECR no lançamento.

Durante o passo de configuração, seu docker file será gerado com base no código da sua aplicação. Observe que ao usar o `bedrock_agentcore_starter_toolkit` para configurar seu agente, ele configura a Observabilidade do AgentCore por padrão, então, para usar o Braintrust, você precisa remover a configuração da Observabilidade do AgentCore conforme explicado abaixo:

<div style="text-align:left">
    <img src="../images/configure.png" width="40%"/>
</div>

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime
from boto3.session import Session
boto_session = Session()
region = boto_session.region_name

agentcore_runtime = Runtime()
agent_name = "strands_agentcore_arize_observability"
response = agentcore_runtime.configure(
    entrypoint="strands_claude.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name=agent_name,
    memory_mode='NO_MEMORY',
    disable_otel=True, 
)
response

## Implantar no AgentCore Runtime

Agora que temos um docker file, vamos lançar o agente no AgentCore Runtime. Isso criará o repositório Amazon ECR e o AgentCore Runtime

<div style="text-align:left">
    <img src="../images/launch.png" width="75%"/>
</div>

In [ ]:
# Set the Space and API keys as headers for authentication
import os
headers = f"space_id={os.environ["ARIZE_SPACE_ID"]},api_key={os.environ["ARIZE_API_KEY"]}"

#Launch Runtime (Configure the runtime to send telemetry data to Arize)
launch_result = agentcore_runtime.launch(
    env_vars={
        "BEDROCK_MODEL_ID": "us.anthropic.claude-3-7-sonnet-20250219-v1:0", # Example model ID
        "OTEL_EXPORTER_OTLP_ENDPOINT": os.environ["ARIZE_ENDPOINT"],  # Use Arize OTEL endpoint
        "OTEL_EXPORTER_OTLP_HEADERS": headers,  # Add Arize auth header
        "DISABLE_ADOT_OBSERVABILITY": "true",   # Bypass Cloudwatch
    }
)
launch_result


## Verificar Status da Implantação

Aguarde o runtime estar pronto antes de invocar:

In [ ]:
import time
status_response = agentcore_runtime.status()
status = status_response.endpoint['status']
end_status = ['READY', 'CREATE_FAILED', 'DELETE_FAILED', 'UPDATE_FAILED']
while status not in end_status:
    time.sleep(10)
    status_response = agentcore_runtime.status()
    status = status_response.endpoint['status']
    print(status)
status

### Invocando o AgentCore Runtime

Finalmente, podemos invocar nosso AgentCore Runtime com um payload

<div style="text-align:left">
    <img src="../images/invoke.png" width=75%"/>
</div>

In [ ]:
invoke_response = agentcore_runtime.invoke({"prompt": "I'm planning a weekend trip to davos. What are the must-visit places and local food I should try?"})

In [ ]:
from IPython.display import Markdown, display
display(Markdown("".join(invoke_response['response'])))

## Visualizar Traces no Arize

Para visualizar os traces:
1. Acesse seu painel do Arize em https://app.arize.com
2. Navegue até seu projeto
3. Clique em "Traces" para visualizar os dados de telemetria

Os traces incluirão:
- Detalhes da invocação do agente
- Chamadas de ferramentas (busca na web)
- Interações com o modelo com latência e uso de tokens
- Payloads de request/response

## Limpeza (Opcional)

Limpar os recursos implantados:

In [ ]:
import boto3

agentcore_control_client = boto3.client(
    'bedrock-agentcore-control',
    region_name=region
)

ecr_client = boto3.client(
    'ecr',
    region_name=region
)

runtime_delete_response = agentcore_control_client.delete_agent_runtime(
    agentRuntimeId=launch_result.agent_id,
)

response = ecr_client.delete_repository(
    repositoryName=launch_result.ecr_uri.split('/')[1],
    force=True
)

## Resumo

Você implantou com sucesso um agente Strands no Amazon Bedrock AgentCore Runtime com observabilidade Arize. A implementação demonstra:
- Integração de agentes Strands com o AgentCore Runtime
- Configuração do OpenTelemetry para enviar traces para o Arize
- Ordem de inicialização adequada para garantir a configuração da telemetria
- Invocação através do SDK e do cliente boto3

O agente está agora executando em um ambiente gerenciado e escalável com observabilidade completa através do Arize.